In [7]:
import numpy as np

In [8]:
class sigmoid:
  def __init__(self): # 인스턴스 메서드 
    self.params = [] # sigmoid 계층에는 학습하는 매개변수가 따로 없으므로 빈 리스트로 초기화한다.
  
  # forward 메서드는 입력 데이터를 받아 순전파를 수행해 활성화 함수의 출력을 반환한다.
  def forward(self, x):
    return 1 / (1 + np.exp(-x)) 
  

In [9]:
class Affine:
  def __init__(self, W, b): # 인스턴스 메서드에서 가중치와 편향을 매개변수로 받아 인스턴스 변수인 params에 저장한다.
    self.params = [W, b]
  
  # forward 메서드는 입력 데이터를 받아 순전파를 수행해 가중치와 편향을 적용한 다음 결과를 반환한다.
  def forward(self, x):
    W, b = self.params
    out = np.dot(x, W) + b
    return out

### 신경망 '추론' 계층 구현  

In [21]:
# Affine - Sigmoid - Affine 계층으로 이어지는 추론 계층 구현
class TwoLayerNet:
  def __init__(self, input_size, hidden_size, output_size):
    I, H, O = input_size, hidden_size, output_size # 인수로 받은 input_size, hidden_size, output_size를 인스턴스 변수로 저장한다

    # 가중치와 편향 초기화
    # 정규분포를 따르는 난수로 가중치와 편향을 랜덤하게 초기화한다.
    W1 = np.random.randn(I, H)
    b1 = np.random.randn(H) # Hidden Layer의 뉴런 수만큼 편향을 생성한다.
    W2 = np.random.randn(H, O)
    b2 = np.random.randn(O) # Output Layer의 뉴런 수만큼 편향을 생성한다.

    # 계층 생성 Affine - Sigmoid - Affine
    self.layers = [
      Affine(W1, b1),
      sigmoid(),
      Affine(W2, b2)
    ]

    # 모든 가중치를 리스트에 모은다.
    self.params = []
    for layer in self.layers:
      self.params += layer.params

  # forward 메서드는 인수 x를 받아 신경망의 순전파를 수행해 결과를 반환한다.
  def forward(self, x):
    for layer in self.layers: # layers 리스트에 저장된 계층을 차례로 forward 메서드를 호출한다.
      x = layer.forward(x) # Affine.forward, sigmoid.forward, Affine.forward 순으로 호출된다.
    return x # Affine, sigmoid, Affine 순전파를 거친 결과를 반환한다.

In [22]:
x = np.random.randn(10, 2) # 입력 데이터 10개, 2차원
model = TwoLayerNet(2, 4, 3) # input_size=2, hidden_size=4, output_size=3
s = model.forward(x) # 순전파

In [23]:
print(s) # 출력 결과

[[ 1.44452846  0.11359869  2.17044445]
 [ 1.31707952 -0.33601799  2.07216614]
 [ 1.1378958  -0.43097249  1.83885761]
 [ 0.81342465 -0.36275661  1.22377835]
 [ 1.2726351  -0.43827835  2.01298757]
 [ 2.09625922  0.73383062  2.61603187]
 [ 2.21863922  0.66939132  2.67622048]
 [ 0.20789278 -0.91086805  0.61164455]
 [ 1.50386225 -0.47023836  2.23881595]
 [ 1.24051014  1.02210585  1.97393349]]


### 신경망 추론과 학습을 포함한 계층 구현  

In [ ]:
# 순전파와 역전파를 모두 구현한 sigmoid 계층
class sigmoid:
  def __init__(self):
      self.params, self.grads = [], [] # params와 grads를 빈 리스트로 초기화한다. params는 가중치 매개변수를, grads는 기울기를 저장한다.
      self.out = None # 순전파의 출력을 보관한다.
  
  # 순전파 메서드는 입력 데이터를 받아 활성화 함수를 적용한 다음 결과를 반환한다.
  def forward(self, x):
      out = 1 / (1 + np.exp(-x))
      self.out = out
      return out # 순전파의 출력을 out에 저장하고 반환한다.
    
  # 역전파를 계산할 때 out 변수를 사용한다.
  def backward(self, dout):
      dx = dout * (1.0 - self.out) * self.out # y = 1 / (1 + exp(-x))의 미분은 y(1-y)이다.
      return dx
  

In [ ]:
# 순전파와 역전파를 모두 구현한 Affine 계층
class Affine:
  def __init__(self, W, b):
      self.params = [W, b] # 가중치와 편향을 params에 저장한다.
      self.grads = [np.zeros_like(W), np.zeros_like(b)] # 가중치와 편향의 기울기를 grads에 저장한다.
      self.x = None # 순전파의 입력을 보관한다.
  
  # 순전파 메서드는 입력 데이터를 받아 가중치와 편향을 적용한 다음 결과를 반환한다.
  def forward(self, x):
      W, b = self.params
      out = np.dot(x, W) + b
      self.x = x
      return out # 순전파의 입력을 x에 저장하고 결과를 반환한다.
  
  # 역전파 메서드는 상류에서 전해지는 기울기 dout을 받아 가중치와 편향의 기울기를 계산하고 하류로 전달한다.
  def backward(self, dout):
      W, b = self.params
      dx = np.dot(dout, W.T) # dx = dout * W^T, dL/dy * W^T, 입력 x에 대한 상류 값의 미분으로 이전 층으로 전파해야 할 값이다
      dW = np.dot(self.x.T, dout) # dW = x^T * dout, X^T * dL/dy, 가중치 W에 대한 손실 함수의 기울기이다.
      db = np.sum(dout, axis=0) # # db = dL/dy의 각 원소의 총합이다, axis = 0은 행 방향으로 더한다
      self.grads[0][...] = dW # 가중치의 기울기를 grads에 저장한다.
      self.grads[1][...] = db # 편향의 기울기를 grads에 저장한다.
      
      return dx # 역전파의 결과 x에 대한 손실함수의 기울기를 반환한다.

In [ ]:
def softmax(x):
  # 입력 데이터가 2차원 배열일 때는 배치용 softmax 함수를 적용한다.
  if x.ndim == 2:
    x = x - x.max(axis=1, keepdims=True)
    x = np.exp(x)
    x /= x.sum(axis=1, keepdims=True)
    
  # 입력 데이터가 1차원 배열일 때는 기본 softmax 함수를 적용한다.
  elif x.ndim == 1:
    x = x - np.max(x)
    x = np.exp(x) / np.sum(np.exp(x))
  return x

def cross_entropy_error(y, t): # y는 신경망의 출력(예측 값), t는 정답 레이블
  # 데이터가 하나인 경우, 데이터의 형상을 바꿔준다.
  if y.ndim == 1:
    t = t.reshape(1, t.size)
    y = y.reshape(1, y.size)
    
  # 정답 레이블 크기와 예측 값 크기가 같으면 정답 레이블의 인덱스로 반환한다.
  if t.size == y.size:
    t = t.argmax(axis=1) # 원 핫 인코딩된 정답 레이블을 인덱스로 변환한다. 즉 1(정답)의 위치를 반환한다.
          
  batch_size = y.shape[0] # 배치 크기를 구한다.
  return -np.sum(np.log(y[np.arange(batch_size), t] + 1e-7)) / batch_size # 평균 교차 엔트로피 오차를 반환한다.

In [ ]:
# 순전파와 역전파를 모두 구현한 Softmax with Loss 계층
class SoftmaxWithLoss:
  def __init__(self):
      self.params, self.grads = [], []
      self.y = None # softmax의 출력
      self.t = None # 정답 레이블
      
  # 순전파 메서드는 입력 데이터와 정답 레이블을 받아 소프트맥스 함수와 교차 엔트로피 오차를 적용한 다음 결과를 반환한다.
  def forward(self, x, t):
      self.t = t
      self.y = softmax(x)
      
      # 정답 레이블이 원-핫 벡터일 경우 정답 레이블의 인덱스로 변환한다.
      if self.t.size == self.y.size:
          self.t = self.t.argmax(axis=1)
          
      loss = cross_entropy_error(self.y, self.t)
      return loss # 소프트맥스 함수와 교차 엔트로피 오차를 적용한 결과를 반환한다.
  
  # 역전파 메서드는 상류에서 전해지는 기울기 dout을 받아 소프트맥스 함수와 교차 엔트로피 오차의 기울기를 계산하고 하류로 전달한다.
  def backward(self, dout=1):
      batch_size = self.t.shape[0]
      
      # 정답 레이블이 원-핫 벡터일 경우 정답 레이블의 인덱스로 변환한다.
      if self.t.size == self.y.size:
          dx = self.y.copy()
          dx[np.arange(batch_size), self.t] -= 1
      else:
          dx = self.y - self.t
          
      dx *= dout
      dx = dx / batch_size
      
      return dx # 소프트맥스 함수와 교차 엔트로피 오차의 기울기를 반환한다.